# Импорты библиотек и загрузки данных

In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl

import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score

import joblib

import os

import numpy as np
from pathlib import Path

## Пути

In [2]:
path_train = "data\\train\\"
path_test = "data\\test\\"
path_sample_submit = "data\\sample_submit.parquet"
path_summit = "data\\submission\\"

path_save_model_all_feature = "data\model\model_baseline_all_feature\\"
path_save_model_part_feature = "data\model\model_baseline_part_feature\\"
path_save_model_minimum_feature = "data\model\model_minimum_feature\\"
path_save_model_with_extra_feature = "data\model\model_extra_feature\\"

## Данные

In [3]:
train_m = pd.read_parquet(f"{path_train}train_main_features.parquet")
train_e = pd.read_parquet(f"{path_train}train_extra_features.parquet")
train_t = pd.read_parquet(f"{path_train}train_target.parquet")

In [4]:
train = train_m.merge(train_t, on="customer_id", how="left")

In [5]:
train_x, val_x, train_y, val_y = train_test_split(train_m, train_t, test_size=0.2, random_state=42)

In [6]:
test_m = pd.read_parquet(f"{path_test}test_main_features.parquet")
test_e = pd.read_parquet(f"{path_test}test_extra_features.parquet")

In [7]:
sample_submit = pd.read_parquet(path_sample_submit)

# Вспомогательные функции

In [8]:
num_cols = list(train_m.columns[train_m.columns.str.startswith("num")])
cat_cols = list(train_m.columns[train_m.columns.str.startswith("cat")])
target_cols = [col for col in train_t.columns if col != "customer_id"]

In [9]:
def validation(model, val_y):
    predict = model.predict_proba(val_x)[:,1]

    score = roc_auc_score(val_y, predict)

    print(f"{score:.6f}\n")

    return score

In [10]:
def loading_unloading_wrapper(path_model, code_study_f, params, train_y, val_y, cat_cols = cat_cols):
    print(f"{val_y.name}")
    if not os.path.exists(path_model):
        model = code_study_f(train_y, val_y, cat_cols, **params)
        joblib.dump(model, path_model)
    else:
        model = joblib.load(path_model)

    score = validation(model, val_y)

    return score

In [11]:
def study_model_all_feature(train_y, val_y, cat_cols, **params):
    model = lgb.LGBMClassifier(**params)

    model.fit(train_x,
            train_y,
            eval_set=[(val_x, val_y)],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=100),
            ],
            )

    return model

In [12]:
def feature_importance(model):
    summ_imp = sum(model.feature_importances_)
    feat_imp_df = pd.DataFrame({
        "feature": model.feature_name_,
        "importance": model.feature_importances_ / summ_imp
    })

    return feat_imp_df.sort_values("importance", ascending=False)

In [13]:
def all_feature_importance(paths):
    model = joblib.load(paths[0])
    feature_importance_table = pd.DataFrame({
        "feature": model.feature_name_,
        "importance": 0.0
    }).set_index("feature")

    for path in paths:
        model = joblib.load(path)
        feature_imp = feature_importance(model)
        for row in feature_imp.itertuples():
            feature_importance_table.loc[row.feature, "importance"] += row.importance

    feature_importance_table["importance"] /= len(paths)

    return feature_importance_table.reset_index().sort_values("importance", ascending=False)


# Начальное исследование данных (вход\выход)

In [12]:
sample_submit.head()

,customer_id,predict_1_1,predict_1_2,predict_1_3,predict_1_4,predict_1_5,predict_2_1,predict_2_2,predict_2_3,predict_2_4,...,predict_8_3,predict_9_1,predict_9_2,predict_9_3,predict_9_4,predict_9_5,predict_9_6,predict_9_7,predict_9_8,predict_10_1
0,1750001,-4.559634,-5.490853,-3.682293,-3.696798,-6.377251,-4.698343,-3.925055,-6.864407,-4.644582,...,-3.969470,-5.522536,-2.835501,-3.592803,-5.626597,-4.600432,-0.826411,-2.292409,-4.566064,-0.624437
1,1750002,-4.525017,-5.424643,-3.600475,-3.663936,-6.481639,-4.698258,-3.878170,-6.936742,-4.645305,...,-3.882064,-5.586788,-2.723239,-3.643311,-5.593574,-4.575498,-0.778346,-2.265025,-4.476992,-0.654095
2,1750003,-4.236281,-5.217968,-3.718711,-3.652767,-6.019878,-5.293401,-3.742374,-7.368946,-4.510376,...,-3.915707,-5.821074,-3.501836,-3.764656,-5.877993,-5.007001,-1.049139,-2.805007,-5.363468,-0.362334
3,1750004,-5.107494,-5.747457,-3.870310,-3.988364,-6.613195,-4.781696,-4.434008,-7.223677,-4.649338,...,-4.015814,-5.678406,-2.961717,-3.568433,-5.278345,-4.324995,-0.696825,-2.157653,-4.039829,-0.652303
4,1750005,-4.482090,-5.395573,-3.571668,-3.641002,-6.268344,-4.670509,-3.762274,-6.943392,-4.550145,...,-3.632175,-5.412687,-2.838579,-3.566938,-5.623631,-4.545470,-0.736783,-2.241710,-4.585213,-0.668844


У нас 41 товар\услуга\категория, для которой мы должны выдать число. Это число может быть вероятность покупки, может что-либо другое, как на примере выше, главное это позиция при отранжировании этих числовых значений, так как метрика ROC-AUC. 

In [13]:
train_m.head()

,customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,...,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
0,1000001,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,NaN,-0.445279,NaN,NaN,-0.107666,-0.418616,NaN,NaN
1,1000002,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,...,-0.031281,-0.046146,-0.10217,1.550722,NaN,NaN,-0.170724,-0.805771,-0.397803,-0.373734
2,1000003,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,4.0,...,-0.031281,-0.046146,NaN,-0.475778,NaN,NaN,-0.170724,-0.602005,-0.397803,-0.373734
3,1000004,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,3.0,...,-0.031281,-0.046146,NaN,-0.475778,0.111196,0.116695,NaN,-0.724265,NaN,NaN
4,1000005,1.0,2.0,0.0,1.0,0.0,3.0,0.0,0.0,2.0,...,NaN,-0.046146,NaN,NaN,NaN,NaN,-0.107666,NaN,-0.397803,-0.373734


У нас есть большая часть категориальных и еще большая числовых признаков. Мы не знаем, какие признаки за что отвечают, но мы можем отследить поведение пользователей отраженное в данных признаках.

Таким образом нам необходимо: поработать над признаками, обучить модель на multi-label задачу, так чтобы для каждой категории выдавалось значение, которое бы выдавало значение, которое можно было бы интерпритировать как вероятность клика, но не обязательно по шкале 0 - 100%, а любой шкале, где будет значение негатива, меньше значения позитива.

Также проверим какой вид данных у test

In [14]:
test_m.head()

,customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,...,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
0,1750001,1.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,0.062409,-0.475778,NaN,NaN,0.018451,-0.765018,-0.397803,-0.373734
1,1750002,0.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,NaN,-0.475778,NaN,NaN,-0.170724,-0.357487,-0.397803,-0.373734
2,1750003,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,NaN,NaN,-0.455446,NaN,NaN,0.018451,-0.785394,NaN,NaN
3,1750004,1.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,0.054572,-0.475778,NaN,NaN,NaN,-0.520499,NaN,NaN
4,1750005,0.0,1.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,NaN,1.533778,1.318607,NaN,-0.170724,0.314939,NaN,NaN


# Исследование на данных

### Распределение столбцов

Выделим числовые и категоривальные признаки.

In [ ]:
num_cols = list(train_m.columns[train_m.columns.str.startswith("num")])
cat_cols = list(train_m.columns[train_m.columns.str.startswith("cat")])

print(f"Количество числовых столбцов: {len(num_cols)}, количество категориальных столбцов: {len(cat_cols)}")

Количество числовых столбцов: 132, количество категориальных столбцов: 67


Выделим целевые признаки

In [ ]:
target_cols = [col for col in train_t.columns if col != "customer_id"]

## Проверка исключения полупустых признаков

### Рассмотрим NaN

In [ ]:
train_m.isnull().sum().reset_index(name="kol_null").sort_values("kol_null", ascending=False).rename(columns={"index": "feature"})

,feature,kol_null
110,num_feature_43,749166
121,num_feature_54,748897
131,num_feature_64,748761
101,num_feature_34,748228
185,num_feature_118,747082
...,...,...
8,cat_feature_8,0
4,cat_feature_4,0
5,cat_feature_5,0
6,cat_feature_6,0


Думаю признаки у которых более 30% пропусков не имеет большого смысла рассматривать, сконцетрируемся на более полезных, но перед этим я хочу посмотреть таблицу корреляций до и после, и как измениться score baseline LigthGBM модели до и после удаления.

### Параметры

In [14]:
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "n_estimators": 3000,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

### Обучение со всеми признаками

Но чтобы обучить модель, надо чтобы train_y и val_y были чем-то заполнены, мы покачто заполним все NaN нулями.

In [46]:
train_m_ = train_m.copy()
train_m_ = train_m_.fillna(0)

In [47]:
train_x, val_x, train_y, val_y = train_test_split(train_m_, train_t, test_size=0.2, random_state=42)

In [ ]:
scores = []
for target in target_cols:
    train_y_ = train_y[target]
    val_y_ = val_y[target]
    score = loading_unloading_wrapper(Path(f"{path_save_model_all_feature}{target}.pkl"), study_model_all_feature, params, train_y_, val_y_, cat_cols=cat_cols)
    scores.append(score)

print(f"Macro_average: {sum(scores) / len(scores)}")

target_1_1
0.897839

target_1_2
0.775496

target_1_3
0.855637

target_1_4
0.811277

target_1_5
0.826816

target_2_1
0.790070

target_2_2
0.924425

target_2_3
0.755610

target_2_4
0.704823

target_2_5
0.717978

target_2_6
0.705153

target_2_7
0.692178

target_2_8
0.764551

target_3_1
0.670214

target_3_2
0.901591

target_3_3
0.725486

target_3_4
0.907846

target_3_5
0.955483

target_4_1
0.834830

target_5_1
0.715882

target_5_2
0.681858

target_6_1
0.692519

target_6_2
0.691101

target_6_3
0.726206

target_6_4
0.819231

target_6_5
0.852275

target_7_1
0.787505

target_7_2
0.825147

target_7_3
0.761845

target_8_1
0.969504

target_8_2
0.822077

target_8_3
0.865812

target_9_1
0.752077

target_9_2
0.815057

target_9_3
0.663900

target_9_4
0.887346

target_9_5
0.811878

target_9_6
0.675817

target_9_7
0.744274

target_9_8
0.918346

target_10_1
0.742363

Macro_average: 0.7912029834903


### Перед удалением признаков, я хочу посмотреть какие признаки какой вклад совершили и корреляцию с признаками, имеющими большое число пропусков

In [22]:
paths = [Path(f"{path_save_model_all_feature}{target}.pkl") for target in target_cols]
feature_imp = all_feature_importance(paths)
feature_imp

,feature,importance
39,cat_feature_39,0.195515
108,num_feature_41,0.032217
52,cat_feature_52,0.030294
143,num_feature_76,0.021190
184,num_feature_117,0.019425
...,...,...
99,num_feature_32,0.000000
81,num_feature_14,0.000000
147,num_feature_80,0.000000
137,num_feature_70,0.000000


In [23]:
weak_feature = set(feature_imp[feature_imp["importance"] < 0.00005]["feature"])
print(len(weak_feature))

21


Признаки с 30%+ пропусков

In [24]:
kol_string = train_m.shape[0]
per30_string = int(kol_string * 0.3)

more_null_feature = set(train_m.columns[train_m.isnull().sum() > per30_string])
print(len(more_null_feature))

79


In [25]:
print("Количество столбцов")
print(f"Только в слабых признаках: {len(weak_feature - more_null_feature)}")
print(f"Только в NaN признаках: {len(more_null_feature - weak_feature)}")
print(f"В обоих: {len(more_null_feature & weak_feature)}")

Количество столбцов
Только в слабых признаках: 17
Только в NaN признаках: 75
В обоих: 4


Результаты меня довольно сильно удивили, многие столбцы которые имеют много NaN видимо являются информативными. Попробуем обучить несколько моделей, чтобы понять более точно.

### Обучение без NaN признаков

In [26]:
train_m_ = train_m.drop(columns = more_null_feature)
train_m_ = train_m_.fillna(0)

In [27]:
train_x, val_x, train_y, val_y = train_test_split(train_m_, train_t, test_size=0.2, random_state=42)

In [ ]:
scores = []
for target in target_cols:
    train_y_ = train_y[target]
    val_y_ = val_y[target]
    score = loading_unloading_wrapper(Path(f"{path_save_model_part_feature}{target}.pkl"), study_model_all_feature, params, train_y_, val_y_, cat_cols)
    scores.append(score)

print(f"Macro_average: {sum(scores) / len(scores)}")

target_1_1
[100]	valid_0's auc: 0.865401
0.866690

target_1_2
[100]	valid_0's auc: 0.761104
0.770569

target_1_3
[100]	valid_0's auc: 0.849234
[200]	valid_0's auc: 0.848393
0.849581

target_1_4
[100]	valid_0's auc: 0.79749
[200]	valid_0's auc: 0.795952
0.797490

target_1_5
[100]	valid_0's auc: 0.814787
0.816765

target_2_1
[100]	valid_0's auc: 0.788895
0.797126

target_2_2
[100]	valid_0's auc: 0.904275
[200]	valid_0's auc: 0.904369
0.904809

target_2_3
[100]	valid_0's auc: 0.734089
0.747796

target_2_4
[100]	valid_0's auc: 0.693116
0.701630

target_2_5
[100]	valid_0's auc: 0.709697
[200]	valid_0's auc: 0.7111
0.712735

target_2_6
[100]	valid_0's auc: 0.703699
[200]	valid_0's auc: 0.702658
0.704685

target_2_7
[100]	valid_0's auc: 0.715845
[200]	valid_0's auc: 0.701533
0.790283

target_2_8
[100]	valid_0's auc: 0.577447
[200]	valid_0's auc: 0.351927
0.589695

target_3_1
[100]	valid_0's auc: 0.647208
[200]	valid_0's auc: 0.647567
[300]	valid_0's auc: 0.647352
0.647859

target_3_2
[100]	va

### Обучение без NaN признаков и слабых признаков

In [32]:
feature_drop = more_null_feature | (weak_feature - more_null_feature)

In [33]:
train_m_ = train_m.drop(columns = feature_drop)
train_m_ = train_m_.fillna(0)

In [34]:
train_x, val_x, train_y, val_y = train_test_split(train_m_, train_t, test_size=0.2, random_state=42)

In [ ]:
cat_cols_ = list(set(cat_cols) - feature_drop)

In [ ]:
scores = []
for target in target_cols:
    train_y_ = train_y[target]
    val_y_ = val_y[target]
    score = loading_unloading_wrapper(Path(f"{path_save_model_minimum_feature}{target}.pkl"), study_model_all_feature, params, train_y_, val_y_, cat_cols = cat_cols_)
    scores.append(score)

print(f"Macro_average: {sum(scores) / len(scores)}")

target_1_1
[100]	valid_0's auc: 0.862427
0.863711

target_1_2
[100]	valid_0's auc: 0.764345
0.772152

target_1_3
[100]	valid_0's auc: 0.848689
[200]	valid_0's auc: 0.848381
0.849403

target_1_4
[100]	valid_0's auc: 0.795517
[200]	valid_0's auc: 0.793534
0.796087

target_1_5
[100]	valid_0's auc: 0.803858
0.806228

target_2_1
[100]	valid_0's auc: 0.783242
0.790010

target_2_2
[100]	valid_0's auc: 0.904431
[200]	valid_0's auc: 0.904839
0.904897

target_2_3
[100]	valid_0's auc: 0.728833
0.744729

target_2_4
[100]	valid_0's auc: 0.694033
0.699088

target_2_5
[100]	valid_0's auc: 0.694086
[200]	valid_0's auc: 0.693004
0.695551

target_2_6
[100]	valid_0's auc: 0.699591
[200]	valid_0's auc: 0.701331
[300]	valid_0's auc: 0.703627
[400]	valid_0's auc: 0.702439
0.704468

target_2_7
[100]	valid_0's auc: 0.638135
[200]	valid_0's auc: 0.73377
[300]	valid_0's auc: 0.738551
[400]	valid_0's auc: 0.755054
[500]	valid_0's auc: 0.75861
0.761088

target_2_8
[100]	valid_0's auc: 0.577313
0.748269

target_3_

### Обучение с extra feature

In [15]:
train = train_m.merge(train_e, on="customer_id", how="left")

In [22]:
drop_feature = list(train.isnull().sum().sort_values(ascending=False)[:600].index)

In [25]:
train_ = train.fillna(-999)

In [26]:
train_x, val_x, train_y, val_y = train_test_split(train_, train_t, test_size=0.2, random_state=42)

In [27]:
cat_cols_ = list(train_.columns[train_.columns.str.startswith("cat")])

In [28]:
scores = []
for target in target_cols:
    train_y_ = train_y[target]
    val_y_ = val_y[target]
    score = loading_unloading_wrapper(Path(f"{path_save_model_with_extra_feature}{target}.pkl"), study_model_all_feature, params, train_y_, val_y_, cat_cols = cat_cols_)
    scores.append(score)

print(f"Macro_average: {sum(scores) / len(scores)}")

target_1_1
[100]	valid_0's auc: 0.902213
[200]	valid_0's auc: 0.900936
0.902630

target_1_2
[100]	valid_0's auc: 0.793385
0.794842

target_1_3
[100]	valid_0's auc: 0.868987
[200]	valid_0's auc: 0.868935
0.869507

target_1_4
[100]	valid_0's auc: 0.825695
[200]	valid_0's auc: 0.825463
0.826657

target_1_5
[100]	valid_0's auc: 0.865742
[200]	valid_0's auc: 0.872226
[300]	valid_0's auc: 0.872155
[400]	valid_0's auc: 0.869336
0.874227

target_2_1
[100]	valid_0's auc: 0.812991
0.813762

target_2_2
[100]	valid_0's auc: 0.935548
[200]	valid_0's auc: 0.935979
0.936142

target_2_3
[100]	valid_0's auc: 0.755543
0.771951

target_2_4
[100]	valid_0's auc: 0.72655
0.730063

target_2_5
[100]	valid_0's auc: 0.725351
[200]	valid_0's auc: 0.731256
[300]	valid_0's auc: 0.728443
0.732747

target_2_6
[100]	valid_0's auc: 0.721986
0.729036

target_2_7
[100]	valid_0's auc: 0.583129
[200]	valid_0's auc: 0.719289
[300]	valid_0's auc: 0.775681
[400]	valid_0's auc: 0.795892
0.799060

target_2_8
[100]	valid_0's au

### Проверим гипотезы

In [30]:
paths = []
for target in target_cols:
    paths.append(f"{path_save_model_with_extra_feature}{target}.pkl")

feature_imp_t = all_feature_importance(paths)

In [44]:
feature_imp_t[feature_imp_t["importance"] < 0.0001]

,feature,importance
350,num_feature_353,0.000100
1209,num_feature_1519,0.000098
1183,num_feature_1486,0.000098
373,num_feature_383,0.000098
1168,num_feature_1468,0.000098
...,...,...
633,num_feature_744,0.000000
1482,num_feature_1901,0.000000
1840,num_feature_2373,0.000000
1810,num_feature_2335,0.000000


Огромное число слабых признаков, удалим их, облегчив работу по обучению, но это в следующем файле, этот я считаю уже переполненным.